# Exective summary of this TD

## Objectives

This TD aims at establishing a proper workflow for benchmarking the performance of different machine learning models using training dataset, and testing different models using testing dataset and make a submission on Kaggle. More specifically, you will need to:
- Understand how to use the utility function to read the data, and design a cross validation pipeline.
- Develop one fault detection model for each motor. You need to try different features, models, and conclude the best models from your experiment results.
- Use the best models to predict the labels for the testing dataset, and make a submission on Kaggle.

# Please put the group members here

- Óscar MARTÍNEZ ZAMORA
- Ahmed-Wassim BENZERGA
- Yiling ZHUANG

# Motor 1

You are supposed to experiment different classification-based fault detection models to get best F1 score. Please use the k-fold cross-validation to calculate the best F1 score. You are free to try different models, whether they are discussed in the class or not. 

Please report all the models you tried, features used, how to you tune their hyperparameters, and the corresponding F1 score. Use one subsection per model. Please note that if you would like to tune the hyperparameter, you can use the `GridSearchCv` function in scikit-learn, but you should use it only on the training dataset.

In [1]:
import sys
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

utility_path = 'kaggle_data_challenge/'
sys.path.insert(1, utility_path)
from utility import read_all_test_data_from_path, run_cv_one_motor, FaultDetectReg, extract_selected_feature

def compensate_seq_bias(df: pd.DataFrame):
    df['temperature'] = df['temperature'] - df['temperature'].iloc[0]
    df['voltage'] = df['voltage'] - df['voltage'].iloc[0]
    df['position'] = df['position'] - df['position'].iloc[0]

def remove_outliers(df: pd.DataFrame):
    df['temperature'] = df['temperature'].where(df['temperature'] <= 100, np.nan)
    df['temperature'] = df['temperature'].where(df['temperature'] >= 0, np.nan)
    df['temperature'] = df['temperature'].ffill()        

    df['voltage'] = df['voltage'].where(df['voltage'] >= 6000, np.nan)
    df['voltage'] = df['voltage'].where(df['voltage'] <= 9000, np.nan)
    df['voltage'] = df['voltage'].ffill()        

    df['position'] = df['position'].where(df['position'] >= 0, np.nan)
    df['position'] = df['position'].where(df['position'] <= 1000, np.nan)
    df['position'] = df['position'].ffill()

def cal_diff(df: pd.DataFrame, n_int: int):
    df['temperature_diff'] = df['temperature'].diff(n_int)
    df['voltage_diff'] = df['voltage'].diff(n_int)
    df['position_diff'] = df['position'].diff(n_int)

def pre_processing(df: pd.DataFrame):
    remove_outliers(df)
    compensate_seq_bias(df)
    cal_diff(df, 10)
    df.fillna(0, inplace=True)


base_dictionary = 'data/training_data/'
df_data = read_all_test_data_from_path(base_dictionary, pre_processing, is_plot=False)

df_data_experiment = df_data

# Features for classification models (RF, GB)
feature_list_all = ['data_motor_1_temperature', 'data_motor_1_voltage',
                    'data_motor_2_temperature', 'data_motor_2_voltage',
                    'data_motor_3_temperature', 'data_motor_3_voltage',
                    'data_motor_4_temperature', 'data_motor_4_voltage',
                    'data_motor_5_temperature', 'data_motor_5_voltage',
                    'data_motor_6_temperature', 'data_motor_6_voltage',
                    'data_motor_1_position_diff', 'data_motor_1_temperature_diff', 'data_motor_1_voltage_diff',
                    'data_motor_2_position_diff', 'data_motor_2_temperature_diff', 'data_motor_2_voltage_diff',
                    'data_motor_3_position_diff', 'data_motor_3_temperature_diff', 'data_motor_3_voltage_diff',
                    'data_motor_4_position_diff', 'data_motor_4_temperature_diff', 'data_motor_4_voltage_diff',
                    'data_motor_5_position_diff', 'data_motor_5_temperature_diff', 'data_motor_5_voltage_diff',
                    'data_motor_6_position_diff', 'data_motor_6_temperature_diff', 'data_motor_6_voltage_diff']

# Features for AAKR regression model (includes position, voltage, time + diffs)
feature_list_reg = ['time',
                    'data_motor_1_position', 'data_motor_1_temperature', 'data_motor_1_voltage',
                    'data_motor_2_position', 'data_motor_2_temperature', 'data_motor_2_voltage',
                    'data_motor_3_position', 'data_motor_3_temperature', 'data_motor_3_voltage',
                    'data_motor_4_position', 'data_motor_4_temperature', 'data_motor_4_voltage',
                    'data_motor_5_position', 'data_motor_5_temperature', 'data_motor_5_voltage',
                    'data_motor_6_position', 'data_motor_6_temperature', 'data_motor_6_voltage']

n_cv = 5
window_size = 50
sample_step = 20

## Model 1: AAKR (Auto-Associative Kernel Regression)

AAKR is a **regression-based** fault detection approach. Instead of directly classifying fault vs. normal, it:
1. Trains a kernel regression model on **normal data only** to learn the expected temperature behavior.
2. At prediction time, compares the predicted (expected) temperature with the actual measurement.
3. If the residual exceeds a threshold, the sample is flagged as a fault.

We use `FaultDetectReg` from `utility.py` with `pre_trained=False`. The regression model uses a **Nystroem approximation** of the RBF kernel (to avoid memory issues with the exact kernel matrix) followed by a Ridge regressor, wrapped in a `StandardScaler` pipeline.

**Features used:** all 18 raw sensor signals (position, temperature, voltage) from all 6 motors + time.

**Hyperparameters:** `n_components` (Nystroem rank), `gamma` (kernel width), `alpha` (Ridge regularization), plus `threshold` and `window_size` of `FaultDetectReg`.

In [2]:
# Model 1: AAKR — Regression-based fault detection
from sklearn.linear_model import LinearRegression

aakr_pipeline = Pipeline([
    ('standardizer', StandardScaler()),
    ('regressor', LinearRegression())
])

df_x_reg, y_response = extract_selected_feature(df_data_experiment, feature_list_reg, motor_idx=1, mdl_type='reg')
y_label = df_data_experiment['data_motor_1_label']

aakr_window = 10
aakr_sample_step = 1
aakr_pred_lead = 1

detector_aakr = FaultDetectReg(
    reg_mdl=aakr_pipeline,
    pre_trained=False,
    threshold=0.9,
    abnormal_limit=3,
    window_size=aakr_window,
    sample_step=aakr_sample_step,
    pred_lead_time=aakr_pred_lead
)

print("=" * 60)
print("MODEL 1: AAKR (Auto-Associative Kernel Regression) — Motor 1")
print("=" * 60)
print(f"  Regressor:      LinearRegression (StandardScaler pipeline)")
print(f"  Approach:       Regression-based (train on normal data only)")
print(f"  Features:       {len(feature_list_reg)} features (raw signals + time)")
print(f"  Window size:    {aakr_window}")
print(f"  Sample step:    {aakr_sample_step}")
print(f"  Threshold:      0.9 (adaptive, max 1.5)")
print(f"  Abnormal limit: 3")
print(f"  Cross-val:      {n_cv}-fold (by test sequence)")
print("-" * 60)
print("Running cross-validation...\n")

all_result_aakr_m1 = detector_aakr.run_cross_val(
    df_x=df_x_reg,
    y_label=y_label,
    y_response=y_response,
    n_fold=n_cv,
    single_run_result=False
)

print("\n" + "-" * 60)
print("RESULTS — AAKR Motor 1:")
print("-" * 60)
print(all_result_aakr_m1.to_string(index=True))
print(f"\n{'Metric':<12} {'Mean':>8} {'± Std':>8}")
print("-" * 30)
for col in all_result_aakr_m1.columns:
    print(f"{col:<12} {all_result_aakr_m1[col].mean():>8.4f} {all_result_aakr_m1[col].std():>8.4f}")
print("=" * 60)

MODEL 1: AAKR (Auto-Associative Kernel Regression) — Motor 1
  Regressor:      LinearRegression (StandardScaler pipeline)
  Approach:       Regression-based (train on normal data only)
  Features:       19 features (raw signals + time)
  Window size:    10
  Sample step:    1
  Threshold:      0.9 (adaptive, max 1.5)
  Abnormal limit: 3
  Cross-val:      5-fold (by test sequence)
------------------------------------------------------------
Running cross-validation...



 40%|████      | 2/5 [00:04<00:06,  2.10s/it]


KeyboardInterrupt: 

## Model 2: Random Forest

Random Forest is an **ensemble** of decision trees that combines their predictions via majority voting. Key advantages for fault detection:
- Naturally handles class imbalance via `class_weight='balanced'`
- Built-in feature importance scores
- Robust to overfitting thanks to bagging

**Features used:** temperature, voltage, and all diff features (position_diff, temperature_diff, voltage_diff) from all 6 motors (28 features total).

**Hyperparameters tuned via GridSearchCV:** `n_estimators` (number of trees), `max_depth` (tree depth), `min_samples_leaf` (minimum samples per leaf).

In [ ]:
# Model 2: Random Forest
from sklearn.model_selection import StratifiedKFold

inner_cv_rf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

steps_rf = [
    ('standardizer', StandardScaler()),
    ('mdl', RandomForestClassifier(
        class_weight='balanced',
        n_estimators=200,
        random_state=42,
        n_jobs=1
    ))
]
pipeline_rf = Pipeline(steps_rf)

param_grid_rf = {
    'mdl__max_depth': [5, 10, 20, None],
    'mdl__min_samples_leaf': [1, 2, 5]
}

grid_search_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid_rf,
    scoring='f1',
    cv=inner_cv_rf,
    n_jobs=-1,
    error_score=0.0
)

n_combos_rf = len(param_grid_rf['mdl__max_depth']) * len(param_grid_rf['mdl__min_samples_leaf'])
print("=" * 60)
print("MODEL 2: Random Forest — Motor 1")
print("=" * 60)
print(f"  Classifier:     RandomForestClassifier (n_estimators=200)")
print(f"  Class weight:   balanced")
print(f"  Features:       {len(feature_list_all)} features (signals + diffs)")
print(f"  Hyperparams:    max_depth={param_grid_rf['mdl__max_depth']}")
print(f"                  min_samples_leaf={param_grid_rf['mdl__min_samples_leaf']}")
print(f"  Grid search:    {n_combos_rf} combos × 3 inner folds = {n_combos_rf * 3} fits per outer fold")
print(f"  Cross-val:      {n_cv}-fold outer (by test sequence)")
print(f"  Window size:    {window_size}, sample step: {sample_step}")
print("-" * 60)
print("Running cross-validation...\n")

all_result_rf_m1 = run_cv_one_motor(
    motor_idx=1,
    df_data=df_data_experiment,
    mdl=grid_search_rf,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

print("\n" + "-" * 60)
print("RESULTS — Random Forest Motor 1:")
print("-" * 60)
best_rf = grid_search_rf.best_params_ if hasattr(grid_search_rf, 'best_params_') else 'N/A'
print(f"  Best hyperparams: {best_rf}")
print(f"\n{'Metric':<12} {'Mean':>8} {'± Std':>8}")
print("-" * 30)
for col in all_result_rf_m1.columns:
    print(f"{col:<12} {all_result_rf_m1[col].mean():>8.4f} {all_result_rf_m1[col].std():>8.4f}")
print("=" * 60)

MODEL 2: Random Forest — Motor 1
  Classifier:     RandomForestClassifier (n_estimators=200)
  Class weight:   balanced
  Features:       30 features (signals + diffs)
  Hyperparams:    max_depth=[5, 10, 20, None]
                  min_samples_leaf=[1, 2, 5]
  Grid search:    12 combos × 3 inner folds = 36 fits per outer fold
  Cross-val:      5-fold outer (by test sequence)
  Window size:    50, sample step: 20
------------------------------------------------------------
Running cross-validation...

Model for motor 1:
   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.936657        0.0     0.0       0.0
2  0.983816        0.0     0.0       0.0
3  1.000000        1.0     1.0       1.0
4  0.673547        0.0     0.0       0.0


Mean performance metric and standard error:
Accuracy: 0.9188 +- 0.1395
Precision: 0.4000 +- 0.5477
Recall: 0.4000 +- 0.5477
F1 score: 0.4000 +- 0.5477



------------------------------------------------------------
RESULTS — Ra

## Model 3: Gradient Boosting

Histogram-based Gradient Boosting builds an ensemble of weak learners (shallow trees) sequentially, where each new tree corrects the errors of the previous ones. Key advantages:
- Very efficient on large datasets thanks to histogram binning
- Handles class imbalance via `class_weight='balanced'`
- Strong performance on tabular data, often outperforming Random Forest

**Features used:** same 28 features as Random Forest.

**Hyperparameters tuned via GridSearchCV:** `max_iter` (number of boosting rounds), `learning_rate` (step size shrinkage), `max_depth` (tree depth).

In [ ]:
# Model 3: Gradient Boosting
from sklearn.model_selection import StratifiedKFold

inner_cv_gb = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

steps_gb = [
    ('standardizer', StandardScaler()),
    ('mdl', HistGradientBoostingClassifier(
        class_weight='balanced',
        random_state=42
    ))
]
pipeline_gb = Pipeline(steps_gb)

param_grid_gb = {
    'mdl__max_iter': [100, 200],
    'mdl__learning_rate': [0.01, 0.1],
    'mdl__max_depth': [5, 10, None],
    'mdl__min_samples_leaf': [2, 5]
}

grid_search_gb = GridSearchCV(
    estimator=pipeline_gb,
    param_grid=param_grid_gb,
    scoring='f1',
    cv=inner_cv_gb,
    n_jobs=-1,
    error_score=0.0
)

n_combos_gb = len(param_grid_gb['mdl__max_iter']) * len(param_grid_gb['mdl__learning_rate']) * len(param_grid_gb['mdl__max_depth']) * len(param_grid_gb['mdl__min_samples_leaf'])
print("=" * 60)
print("MODEL 3: Gradient Boosting — Motor 1")
print("=" * 60)
print(f"  Classifier:     HistGradientBoostingClassifier")
print(f"  Class weight:   balanced")
print(f"  Features:       {len(feature_list_all)} features (signals + diffs)")
print(f"  Hyperparams:    max_iter={param_grid_gb['mdl__max_iter']}")
print(f"                  learning_rate={param_grid_gb['mdl__learning_rate']}")
print(f"                  max_depth={param_grid_gb['mdl__max_depth']}")
print(f"                  min_samples_leaf={param_grid_gb['mdl__min_samples_leaf']}")
print(f"  Grid search:    {n_combos_gb} combos × 3 inner folds = {n_combos_gb * 3} fits per outer fold")
print(f"  Cross-val:      {n_cv}-fold outer (by test sequence)")
print(f"  Window size:    {window_size}, sample step: {sample_step}")
print("-" * 60)
print("Running cross-validation...\n")

all_result_gb_m1 = run_cv_one_motor(
    motor_idx=1,
    df_data=df_data_experiment,
    mdl=grid_search_gb,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

print("\n" + "-" * 60)
print("RESULTS — Gradient Boosting Motor 1:")
print("-" * 60)
best_gb = grid_search_gb.best_params_ if hasattr(grid_search_gb, 'best_params_') else 'N/A'
print(f"  Best hyperparams: {best_gb}")
print(f"\n{'Metric':<12} {'Mean':>8} {'± Std':>8}")
print("-" * 30)
for col in all_result_gb_m1.columns:
    print(f"{col:<12} {all_result_gb_m1[col].mean():>8.4f} {all_result_gb_m1[col].std():>8.4f}")
print("=" * 60)

MODEL 3: Gradient Boosting — Motor 1
  Classifier:     HistGradientBoostingClassifier
  Class weight:   balanced
  Features:       30 features (signals + diffs)
  Hyperparams:    max_iter=[100, 200]
                  learning_rate=[0.01, 0.1]
                  max_depth=[5, 10, None]
                  min_samples_leaf=[2, 5]
  Grid search:    24 combos × 3 inner folds = 72 fits per outer fold
  Cross-val:      5-fold outer (by test sequence)
  Window size:    50, sample step: 20
------------------------------------------------------------
Running cross-validation...

Model for motor 1:
   Accuracy  Precision    Recall  F1 score
0  0.844291   0.000000  0.000000  0.000000
1  0.711330   0.090666  0.393963  0.147408
2  0.983816   0.000000  0.000000  0.000000
3  1.000000   1.000000  1.000000  1.000000
4  0.673547   0.000000  0.000000  0.000000


Mean performance metric and standard error:
Accuracy: 0.8426 +- 0.1504
Precision: 0.2181 +- 0.4388
Recall: 0.2788 +- 0.4378
F1 score: 0.2295 +- 0.4

## Model 4: Logistic Regression

In [ ]:
# Model 4: Logistic Regression
steps_lr = [
    ('standardizer', StandardScaler()),
    ('mdl', LogisticRegression(
        penalty='elasticnet',
        solver='saga',
        class_weight='balanced',
        max_iter=300,
        tol=1e-2,
        random_state=42
    ))
]
pipeline_lr = Pipeline(steps_lr)

param_grid_lr = {
    'mdl__C': [1, 10],
    'mdl__l1_ratio': [0.5]
}

grid_search_lr = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=param_grid_lr,
    scoring='f1',
    cv=2,
    n_jobs=-1
)

print("Evaluating Logistic Regression for Motor 1")
all_result_lr_m1 = run_cv_one_motor(
    motor_idx=1,
    df_data=df_data_experiment,
    mdl=grid_search_lr,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

Evaluating Logistic Regression for Motor 1
Model for motor 1:
   Accuracy  Precision    Recall  F1 score
0  0.687279   0.000000  0.000000  0.000000
1  0.476688   0.064762  0.540248  0.115659
2  0.970755   0.205128  0.280702  0.237037
3  0.964286   0.000000  0.000000  0.000000
4  0.621305   0.000000  0.000000  0.000000


Mean performance metric and standard error:
Accuracy: 0.7441 +- 0.2178
Precision: 0.0540 +- 0.0890
Recall: 0.1642 +- 0.2428
F1 score: 0.0705 +- 0.1057




## Summary of the results

| Model   | Accuracy | Precision | Recall | F1   |
|---------|----------|-----------|--------|------|
| Model 1 (AAKR) |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 2 (RF)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 3 (GB)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 4 (LR)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|

# Motor 2

You are supposed to experiment different classification-based fault detection models to get best F1 score. Please use the k-fold cross-validation to calculate the best F1 score. You are free to try different models, whether they are discussed in the class or not. 

Please report all the models you tried, features used, how to you tune their hyperparameters, and the corresponding F1 score. Use one subsection per model. Please note that if you would like to tune the hyperparameter, you can use the `GridSearchCv` function in scikit-learn, but you should use it only on the training dataset.

## Model 1: AAKR (Auto-Associative Kernel Regression)

Same regression-based fault detection approach as Motor 1, applied to Motor 2's temperature signal.

In [ ]:
aakr_pipeline = Pipeline([
    ('standardizer', StandardScaler()),
    ('regressor', LinearRegression())
])

df_x_reg, y_response = extract_selected_feature(df_data_experiment, feature_list_reg, motor_idx=2, mdl_type='reg')
y_label = df_data_experiment['data_motor_2_label']

detector_aakr = FaultDetectReg(
    reg_mdl=aakr_pipeline,
    pre_trained=False,
    threshold=0.9,
    abnormal_limit=3,
    window_size=aakr_window,
    sample_step=aakr_sample_step,
    pred_lead_time=aakr_pred_lead
)

print("=" * 60)
print("AAKR (Auto-Associative Kernel Regression) — Motor 2")
print("=" * 60)

all_result_aakr_m2 = detector_aakr.run_cross_val(
    df_x=df_x_reg,
    y_label=y_label,
    y_response=y_response,
    n_fold=n_cv,
    single_run_result=False
)

print()
print("RESULTS — AAKR Motor 2:")
print("-" * 30)
print(all_result_aakr_m2.to_string(index=True))
print()
print(f"{'Metric':<12} {'Mean':>8} {'± Std':>8}")
print("-" * 30)
for col in all_result_aakr_m2.columns:
    print(f"{col:<12} {all_result_aakr_m2[col].mean():>8.4f} {all_result_aakr_m2[col].std():>8.4f}")
print("=" * 60)

AAKR (Auto-Associative Kernel Regression) — Motor 2


100%|██████████| 4/4 [00:01<00:00,  2.52it/s]


RESULTS — AAKR Motor 2:
------------------------------
   Accuracy  Precision   Recall  F1 score
0  0.399236   0.000000  0.00000  0.000000
1  0.806736   0.810495  0.53172  0.642157
2  0.717774   0.068868  0.91250  0.128070
3  0.660714   0.000000  0.00000  0.000000
4  0.872069   0.000000  0.00000  0.000000

Metric           Mean    ± Std
------------------------------
Accuracy       0.6913   0.1823
Precision      0.1759   0.3560
Recall         0.2888   0.4178
F1 score       0.1540   0.2784


## Model 2: Random Forest

Same Random Forest classifier with GridSearchCV as Motor 1, applied to Motor 2. 


In [ ]:
print("Evaluating Random Forest for Motor 2")
all_result_rf_m2 = run_cv_one_motor(
    motor_idx=2,
    df_data=df_data_experiment,
    mdl=grid_search_rf,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

Evaluating Random Forest for Motor 2
Model for motor 2:
   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.673874        0.0     0.0       0.0
2  0.977286        0.0     0.0       0.0
3  1.000000        1.0     1.0       1.0
4  0.677370        0.0     0.0       0.0


Mean performance metric and standard error:
Accuracy: 0.8657 +- 0.1738
Precision: 0.4000 +- 0.5477
Recall: 0.4000 +- 0.5477
F1 score: 0.4000 +- 0.5477




## Model 3: Gradient Boosting

Same Histogram-based Gradient Boosting classifier with GridSearchCV as Motor 1, applied to Motor 2.

In [ ]:
print("Evaluating Gradient Boosting for Motor 2")
all_result_gb_m2 = run_cv_one_motor(
    motor_idx=2,
    df_data=df_data_experiment,
    mdl=grid_search_gb,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

Evaluating Gradient Boosting for Motor 2
Model for motor 2:
   Accuracy  Precision    Recall  F1 score
0  0.999907   0.000000  0.000000  0.000000
1  0.670148   0.134615  0.002105  0.004144
2  0.973027   0.000000  0.000000  0.000000
3  0.978022   0.000000  0.000000  0.000000
4  0.677625   0.000000  0.000000  0.000000


Mean performance metric and standard error:
Accuracy: 0.8597 +- 0.1700
Precision: 0.0269 +- 0.0602
Recall: 0.0004 +- 0.0009
F1 score: 0.0008 +- 0.0019




## Model 4: Logistic Regression

In [ ]:
print("Evaluating Logistic Regression for Motor 2")
all_result_lr_m2 = run_cv_one_motor(
    motor_idx=2,
    df_data=df_data_experiment,
    mdl=grid_search_lr,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

Evaluating Logistic Regression for Motor 2
Model for motor 2:
   Accuracy  Precision   Recall  F1 score
0  0.725833   0.000000  0.00000  0.000000
1  0.417905   0.005493  0.00436  0.004861
2  0.957127   0.000000  0.00000  0.000000
3  0.939560   0.000000  0.00000  0.000000
4  0.687309   0.000000  0.00000  0.000000


Mean performance metric and standard error:
Accuracy: 0.7455 +- 0.2200
Precision: 0.0011 +- 0.0025
Recall: 0.0009 +- 0.0019
F1 score: 0.0010 +- 0.0022




## Summary of the results

| Model   | Accuracy | Precision | Recall | F1   |
|---------|----------|-----------|--------|------|
| Model 1 (AAKR) |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 2 (RF)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 3 (GB)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 4 (LR)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|

# Motor 3

You are supposed to experiment different classification-based fault detection models to get best F1 score. Please use the k-fold cross-validation to calculate the best F1 score. You are free to try different models, whether they are discussed in the class or not.

Please report all the models you tried, features used, how to you tune their hyperparameters, and the corresponding F1 score. Use one subsection per model. Please note that if you would like to tune the hyperparameter, you can use the `GridSearchCv` function in scikit-learn, but you should use it only on the training dataset.

## Model 1: AAKR (Auto-Associative Kernel Regression)

Same regression-based fault detection approach as Motor 1, applied to Motor 3's temperature signal.

In [ ]:
aakr_pipeline = Pipeline([
    ('standardizer', StandardScaler()),
    ('regressor', LinearRegression())
])

df_x_reg, y_response = extract_selected_feature(df_data_experiment, feature_list_reg, motor_idx=3, mdl_type='reg')
y_label = df_data_experiment['data_motor_3_label']

detector_aakr = FaultDetectReg(
    reg_mdl=aakr_pipeline,
    pre_trained=False,
    threshold=0.9,
    abnormal_limit=3,
    window_size=aakr_window,
    sample_step=aakr_sample_step,
    pred_lead_time=aakr_pred_lead
)

print("=" * 60)
print("AAKR (Auto-Associative Kernel Regression) — Motor 3")
print("=" * 60)

all_result_aakr_m3 = detector_aakr.run_cross_val(
    df_x=df_x_reg,
    y_label=y_label,
    y_response=y_response,
    n_fold=n_cv,
    single_run_result=False
)

print()
print("RESULTS — AAKR Motor 3:")
print("-" * 30)
print(all_result_aakr_m3.to_string(index=True))
print()
print(f"{'Metric':<12} {'Mean':>8} {'± Std':>8}")
print("-" * 30)
for col in all_result_aakr_m3.columns:
    print(f"{col:<12} {all_result_aakr_m3[col].mean():>8.4f} {all_result_aakr_m3[col].std():>8.4f}")
print("=" * 60)

AAKR (Auto-Associative Kernel Regression) — Motor 3


100%|██████████| 4/4 [00:01<00:00,  2.58it/s]


RESULTS — AAKR Motor 3:
------------------------------
   Accuracy  Precision    Recall  F1 score
0  0.993015   0.000000  0.000000  0.000000
1  0.941609   0.000000  0.000000  0.000000
2  0.833617   0.107644  0.831325  0.190608
3  0.721154   0.000000  0.000000  0.000000
4  0.754842   0.000000  0.000000  0.000000

Metric           Mean    ± Std
------------------------------
Accuracy       0.8488   0.1170
Precision      0.0215   0.0481
Recall         0.1663   0.3718
F1 score       0.0381   0.0852


## Model 2: Random Forest

Same Random Forest classifier with GridSearchCV as Motor 1, applied to Motor 3.

In [ ]:
print("Evaluating Random Forest for Motor 3")
all_result_rf_m3 = run_cv_one_motor(
    motor_idx=3,
    df_data=df_data_experiment,
    mdl=grid_search_rf,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

Evaluating Random Forest for Motor 3
Model for motor 3:
   Accuracy  Precision  Recall  F1 score
0  1.000000        1.0     1.0       1.0
1  0.997843        0.0     0.0       0.0
2  0.976434        0.0     0.0       0.0
3  1.000000        1.0     1.0       1.0
4  1.000000        1.0     1.0       1.0


Mean performance metric and standard error:
Accuracy: 0.9949 +- 0.0103
Precision: 0.6000 +- 0.5477
Recall: 0.6000 +- 0.5477
F1 score: 0.6000 +- 0.5477




## Model 3: Gradient Boosting

Same Histogram-based Gradient Boosting classifier with GridSearchCV as Motor 1, applied to Motor 3.

In [ ]:
print("Evaluating Gradient Boosting for Motor 3")
all_result_gb_m3 = run_cv_one_motor(
    motor_idx=3,
    df_data=df_data_experiment,
    mdl=grid_search_gb,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

Evaluating Gradient Boosting for Motor 3
Model for motor 3:
   Accuracy  Precision  Recall  F1 score
0  0.999907        0.0     0.0       0.0
1  0.997156        0.0     0.0       0.0
2  0.976434        0.0     0.0       0.0
3  0.888736        0.0     0.0       0.0
4  0.933996        0.0     0.0       0.0


Mean performance metric and standard error:
Accuracy: 0.9592 +- 0.0474
Precision: 0.0000 +- 0.0000
Recall: 0.0000 +- 0.0000
F1 score: 0.0000 +- 0.0000




## Model 4: Logistic Regression

In [ ]:
print("Evaluating Logistic Regression for Motor 3")
all_result_lr_m3 = run_cv_one_motor(
    motor_idx=3,
    df_data=df_data_experiment,
    mdl=grid_search_lr,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

Evaluating Logistic Regression for Motor 3
Model for motor 3:
   Accuracy  Precision  Recall  F1 score
0  0.591172        0.0     0.0       0.0
1  0.543511        0.0     0.0       0.0
2  0.973027        0.0     0.0       0.0
3  0.821429        0.0     0.0       0.0
4  0.493629        0.0     0.0       0.0


Mean performance metric and standard error:
Accuracy: 0.6846 +- 0.2043
Precision: 0.0000 +- 0.0000
Recall: 0.0000 +- 0.0000
F1 score: 0.0000 +- 0.0000




## Summary of the results

| Model   | Accuracy | Precision | Recall | F1   |
|---------|----------|-----------|--------|------|
| Model 1 (AAKR) |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 2 (RF)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 3 (GB)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 4 (LR)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|

# Motor 4

You are supposed to experiment different classification-based fault detection models to get best F1 score. Please use the k-fold cross-validation to calculate the best F1 score. You are free to try different models, whether they are discussed in the class or not.

Please report all the models you tried, features used, how to you tune their hyperparameters, and the corresponding F1 score. Use one subsection per model. Please note that if you would like to tune the hyperparameter, you can use the `GridSearchCv` function in scikit-learn, but you should use it only on the training dataset.

## Model 1: AAKR (Auto-Associative Kernel Regression)

Same regression-based fault detection approach as Motor 1, applied to Motor 4's temperature signal.

In [ ]:
aakr_pipeline = Pipeline([
    ('standardizer', StandardScaler()),
    ('regressor', LinearRegression())
])

df_x_reg, y_response = extract_selected_feature(df_data_experiment, feature_list_reg, motor_idx=4, mdl_type='reg')
y_label = df_data_experiment['data_motor_4_label']

detector_aakr = FaultDetectReg(
    reg_mdl=aakr_pipeline,
    pre_trained=False,
    threshold=0.9,
    abnormal_limit=3,
    window_size=aakr_window,
    sample_step=aakr_sample_step,
    pred_lead_time=aakr_pred_lead
)

print("=" * 60)
print("AAKR (Auto-Associative Kernel Regression) — Motor 4")
print("=" * 60)

all_result_aakr_m4 = detector_aakr.run_cross_val(
    df_x=df_x_reg,
    y_label=y_label,
    y_response=y_response,
    n_fold=n_cv,
    single_run_result=False
)

print()
print("RESULTS — AAKR Motor 4:")
print("-" * 30)
print(all_result_aakr_m4.to_string(index=True))
print()
print(f"{'Metric':<12} {'Mean':>8} {'± Std':>8}")
print("-" * 30)
for col in all_result_aakr_m4.columns:
    print(f"{col:<12} {all_result_aakr_m4[col].mean():>8.4f} {all_result_aakr_m4[col].std():>8.4f}")
print("=" * 60)

AAKR (Auto-Associative Kernel Regression) — Motor 4


100%|██████████| 4/4 [00:02<00:00,  1.45it/s]


RESULTS — AAKR Motor 4:
------------------------------
   Accuracy  Precision    Recall  F1 score
0  0.797635   0.000000  0.000000  0.000000
1  0.488405   0.020532  0.012177  0.015287
2  0.648495   0.061022  0.919540  0.114449
3  0.521978   0.000000  0.000000  0.000000
4  0.303262   0.000000  0.000000  0.000000

Metric           Mean    ± Std
------------------------------
Accuracy       0.5520   0.1847
Precision      0.0163   0.0265
Recall         0.1863   0.4099
F1 score       0.0259   0.0499


## Model 2: Random Forest

Same Random Forest classifier with GridSearchCV as Motor 1, applied to Motor 4.

In [ ]:
print("Evaluating Random Forest for Motor 4")
all_result_rf_m4 = run_cv_one_motor(
    motor_idx=4,
    df_data=df_data_experiment,
    mdl=grid_search_rf,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

Evaluating Random Forest for Motor 4


NameError: name 'grid_search_rf' is not defined

## Model 3: Gradient Boosting

Same Histogram-based Gradient Boosting classifier with GridSearchCV as Motor 1, applied to Motor 4.

In [ ]:
print("Evaluating Gradient Boosting for Motor 4")
all_result_gb_m4 = run_cv_one_motor(
    motor_idx=4,
    df_data=df_data_experiment,
    mdl=grid_search_gb,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

Evaluating Gradient Boosting for Motor 4


NameError: name 'grid_search_gb' is not defined

## Model 4: Logistic Regression

In [ ]:
print("Evaluating Logistic Regression for Motor 4")
all_result_lr_m4 = run_cv_one_motor(
    motor_idx=4,
    df_data=df_data_experiment,
    mdl=grid_search_lr,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

## Summary of the results

| Model   | Accuracy | Precision | Recall | F1   |
|---------|----------|-----------|--------|------|
| Model 1 (AAKR) |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 2 (RF)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 3 (GB)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 4 (LR)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|

# Motor 5

You are supposed to experiment different classification-based fault detection models to get best F1 score. Please use the k-fold cross-validation to calculate the best F1 score. You are free to try different models, whether they are discussed in the class or not.

Please report all the models you tried, features used, how to you tune their hyperparameters, and the corresponding F1 score. Use one subsection per model. Please note that if you would like to tune the hyperparameter, you can use the `GridSearchCv` function in scikit-learn, but you should use it only on the training dataset.

## Model 1: AAKR (Auto-Associative Kernel Regression)

Same regression-based fault detection approach as Motor 1, applied to Motor 5's temperature signal.

In [ ]:
aakr_pipeline = Pipeline([
    ('standardizer', StandardScaler()),
    ('regressor', LinearRegression())
])

df_x_reg, y_response = extract_selected_feature(df_data_experiment, feature_list_reg, motor_idx=5, mdl_type='reg')
y_label = df_data_experiment['data_motor_5_label']

detector_aakr = FaultDetectReg(
    reg_mdl=aakr_pipeline,
    pre_trained=False,
    threshold=0.9,
    abnormal_limit=3,
    window_size=aakr_window,
    sample_step=aakr_sample_step,
    pred_lead_time=aakr_pred_lead
)

print("=" * 60)
print("AAKR (Auto-Associative Kernel Regression) — Motor 5")
print("=" * 60)

all_result_aakr_m5 = detector_aakr.run_cross_val(
    df_x=df_x_reg,
    y_label=y_label,
    y_response=y_response,
    n_fold=n_cv,
    single_run_result=False
)

print()
print("RESULTS — AAKR Motor 5:")
print("-" * 30)
print(all_result_aakr_m5.to_string(index=True))
print()
print(f"{'Metric':<12} {'Mean':>8} {'± Std':>8}")
print("-" * 30)
for col in all_result_aakr_m5.columns:
    print(f"{col:<12} {all_result_aakr_m5[col].mean():>8.4f} {all_result_aakr_m5[col].std():>8.4f}")
print("=" * 60)

## Model 2: Random Forest

Same Random Forest classifier with GridSearchCV as Motor 1, applied to Motor 5.

In [ ]:
print("Evaluating Random Forest for Motor 5")
all_result_rf_m5 = run_cv_one_motor(
    motor_idx=5,
    df_data=df_data_experiment,
    mdl=grid_search_rf,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

## Model 3: Gradient Boosting

Same Histogram-based Gradient Boosting classifier with GridSearchCV as Motor 1, applied to Motor 5.

In [ ]:
print("Evaluating Gradient Boosting for Motor 5")
all_result_gb_m5 = run_cv_one_motor(
    motor_idx=5,
    df_data=df_data_experiment,
    mdl=grid_search_gb,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

## Model 4: Logistic Regression

In [ ]:
print("Evaluating Logistic Regression for Motor 5")
all_result_lr_m5 = run_cv_one_motor(
    motor_idx=5,
    df_data=df_data_experiment,
    mdl=grid_search_lr,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

## Summary of the results

| Model   | Accuracy | Precision | Recall | F1   |
|---------|----------|-----------|--------|------|
| Model 1 (AAKR) |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 2 (RF)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 3 (GB)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 4 (LR)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|

# Motor 6

You are supposed to experiment different classification-based fault detection models to get best F1 score. Please use the k-fold cross-validation to calculate the best F1 score. You are free to try different models, whether they are discussed in the class or not.

Please report all the models you tried, features used, how to you tune their hyperparameters, and the corresponding F1 score. Use one subsection per model. Please note that if you would like to tune the hyperparameter, you can use the `GridSearchCv` function in scikit-learn, but you should use it only on the training dataset.

## Model 1: AAKR (Auto-Associative Kernel Regression)

Same regression-based fault detection approach as Motor 1, applied to Motor 6's temperature signal.

In [ ]:
aakr_pipeline = Pipeline([
    ('standardizer', StandardScaler()),
    ('regressor', LinearRegression())
])

df_x_reg, y_response = extract_selected_feature(df_data_experiment, feature_list_reg, motor_idx=6, mdl_type='reg')
y_label = df_data_experiment['data_motor_6_label']

detector_aakr = FaultDetectReg(
    reg_mdl=aakr_pipeline,
    pre_trained=False,
    threshold=0.9,
    abnormal_limit=3,
    window_size=aakr_window,
    sample_step=aakr_sample_step,
    pred_lead_time=aakr_pred_lead
)

print("=" * 60)
print("AAKR (Auto-Associative Kernel Regression) — Motor 6")
print("=" * 60)

all_result_aakr_m6 = detector_aakr.run_cross_val(
    df_x=df_x_reg,
    y_label=y_label,
    y_response=y_response,
    n_fold=n_cv,
    single_run_result=False
)

print()
print("RESULTS — AAKR Motor 6:")
print("-" * 30)
print(all_result_aakr_m6.to_string(index=True))
print()
print(f"{'Metric':<12} {'Mean':>8} {'± Std':>8}")
print("-" * 30)
for col in all_result_aakr_m6.columns:
    print(f"{col:<12} {all_result_aakr_m6[col].mean():>8.4f} {all_result_aakr_m6[col].std():>8.4f}")
print("=" * 60)

## Model 2: Random Forest

Same Random Forest classifier with GridSearchCV as Motor 1, applied to Motor 6.

In [ ]:
print("Evaluating Random Forest for Motor 6")
all_result_rf_m6 = run_cv_one_motor(
    motor_idx=6,
    df_data=df_data_experiment,
    mdl=grid_search_rf,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

## Model 3: Gradient Boosting

Same Histogram-based Gradient Boosting classifier with GridSearchCV as Motor 1, applied to Motor 6.

In [ ]:
print("Evaluating Gradient Boosting for Motor 6")
all_result_gb_m6 = run_cv_one_motor(
    motor_idx=6,
    df_data=df_data_experiment,
    mdl=grid_search_gb,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

## Model 4: Logistic Regression

In [ ]:
print("Evaluating Logistic Regression for Motor 6")
all_result_lr_m6 = run_cv_one_motor(
    motor_idx=6,
    df_data=df_data_experiment,
    mdl=grid_search_lr,
    feature_list=feature_list_all,
    n_fold=n_cv,
    window_size=window_size,
    sample_step=sample_step,
    single_run_result=False
)

## Summary of the results

| Model   | Accuracy | Precision | Recall | F1   |
|---------|----------|-----------|--------|------|
| Model 1 (AAKR) |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 2 (RF)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 3 (GB)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|
| Model 4 (LR)   |   XX.X%  |   XX.X%   |  XX.X% | XX.X%|

# Prepare final submission

Putting the best models together, we train the best model on **all training data** for each motor, predict labels on the test set, and produce a Kaggle submission CSV.

**Steps:**
1. Read the test data with the same preprocessing pipeline
2. For each motor, retrain the chosen model on all training data (no CV split)
3. Predict fault labels on the test data
4. Save in the required submission format

In [ ]:
# Read test data with same preprocessing
test_dictionary = 'data/testing_data/'
df_test = read_all_test_data_from_path(test_dictionary, pre_processing, is_plot=False)

# Fill NaN (some test sequences may have missing motors)
df_test.fillna(0, inplace=True)

print(f"Test data shape: {df_test.shape}")
print(f"Test sequences: {df_test['test_condition'].nunique()}")

Test data shape: (14157, 44)
Test sequences: 8


In [ ]:
# Read submission template
df_submission = pd.read_csv('sample_submission.csv')

# Choose best model per motor (change based on your CV results)
# Options: grid_search_rf, grid_search_gb, grid_search_lr, or 'aakr'
best_models = {
    1: grid_search_rf,
    2: grid_search_rf,
    3: grid_search_rf,
    4: grid_search_rf,
    5: grid_search_rf,
    6: grid_search_rf,
}

for motor_idx in range(1, 7):
    print(f"Motor {motor_idx}: training on all data and predicting...")

    mdl = best_models[motor_idx]

    if mdl == 'aakr':
        # AAKR: regression-based fault detection
        aakr_pipeline_sub = Pipeline([
            ('standardizer', StandardScaler()),
            ('regressor', LinearRegression())
        ])
        detector = FaultDetectReg(
            reg_mdl=aakr_pipeline_sub, pre_trained=False, threshold=0.9,
            abnormal_limit=3, window_size=aakr_window,
            sample_step=aakr_sample_step, pred_lead_time=aakr_pred_lead
        )

        # Train on all training data
        df_x_reg, y_response = extract_selected_feature(df_data_experiment, feature_list_reg, motor_idx, 'reg')
        y_label = df_data_experiment[f'data_motor_{motor_idx}_label']
        detector.fit(df_x_reg, y_label, y_response)

        # Predict on test data
        df_test_x_reg, y_response_test = extract_selected_feature(df_test, feature_list_reg, motor_idx, 'reg')
        y_pred, _ = detector.predict(df_test_x_reg, y_response_test, complement_truncation=True)
        y_pred = np.array(y_pred)

    else:
        # Classification models (RF, GB, LR)
        df_tr_x, df_tr_y = extract_selected_feature(df_data_experiment, feature_list_all, motor_idx, 'clf')
        X_train, y_train = prepare_sliding_window(df_tr_x, df_tr_y, window_size=window_size, sample_step=sample_step, mdl_type='clf')
        mdl.fit(X_train, y_train)

        df_test_x, _ = extract_selected_feature(df_test, feature_list_all, motor_idx, 'clf')
        X_test = prepare_sliding_window(df_test_x, window_size=window_size, sample_step=sample_step, mdl_type='clf')
        y_pred = mdl.predict(X_test)

    df_submission[f'data_motor_{motor_idx}_label'] = y_pred.astype(int)
    print(f"  -> {int(sum(y_pred))} faults detected out of {len(y_pred)} samples")

print("\nDone!")

NameError: name 'grid_search_rf' is not defined

In [ ]:
# Save submission
import os; os.makedirs('submissions', exist_ok=True)
df_submission.to_csv('submissions/submission.csv', index=False)
print("Submission saved to submissions/submission.csv")
df_submission.head(10)

Submission saved to submission.csv


,idx,data_motor_1_label,data_motor_2_label,data_motor_3_label,data_motor_4_label,data_motor_5_label,data_motor_6_label,test_condition
0,0,-1,-1,-1,-1,-1,-1,20240527_094865
1,1,-1,-1,-1,-1,-1,-1,20240527_094865
2,2,-1,-1,-1,-1,-1,-1,20240527_094865
3,3,-1,-1,-1,-1,-1,-1,20240527_094865
4,4,-1,-1,-1,-1,-1,-1,20240527_094865
5,5,-1,-1,-1,-1,-1,-1,20240527_094865
6,6,-1,-1,-1,-1,-1,-1,20240527_094865
7,7,-1,-1,-1,-1,-1,-1,20240527_094865
8,8,-1,-1,-1,-1,-1,-1,20240527_094865
9,9,-1,-1,-1,-1,-1,-1,20240527_094865
